# 05. Predictions & Submission

In [1]:
import numpy as np
import pandas as pd
import json
import pickle
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import KFold
import lightgbm as lgb
import xgboost as xgb
from scipy.optimize import minimize
from catboost import CatBoostClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
import csv
warnings.filterwarnings('ignore')

In [2]:
DATA_DIR = Path('../data')
MODEL_DIR = Path('../output/models')
MODEL_DIR.mkdir(exist_ok=True)

TEST_INPUT = DATA_DIR / 'test.csv'
SUBMISSION_OUTPUT = 'submission.csv'

## 1. Load Data

In [3]:
# Load training and test data
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

# Load original IBM dataset for augmentation
original_path = f'{DATA_DIR}/WA_Fn-UseC_-Telco-Customer-Churn.csv'
try:
    original = pd.read_csv(original_path)
    print(f"Original IBM dataset shape: {original.shape}")
    
    # Prepare original data - remove customerID and add to training
    original = original.drop('customerID', axis=1)
    original['id'] = range(len(train), len(train) + len(original))
    
    # Augment training data with original
    train_augmented = pd.concat([train, original], ignore_index=True)
    print(f"Augmented train shape: {train_augmented.shape}")
    
    # Use augmented data for training
    train = train_augmented
    print(f"Using augmented training data: {train.shape}")
except Exception as e:
    print(f"Could not load original data: {e}")
    print("Using competition data only.")

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Sample submission shape: {sample_sub.shape}")

Original IBM dataset shape: (7043, 21)
Augmented train shape: (601237, 21)
Using augmented training data: (601237, 21)
Train shape: (601237, 21)
Test shape: (254655, 20)
Sample submission shape: (254655, 2)


In [4]:
# Model parameters - Further tuned for better accuracy
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.01,  # Lower for better accuracy
    'num_leaves': 63,  # Increased capacity
    'max_depth': 8,
    'min_child_samples': 20,
    'feature_fraction': 0.75,
    'bagging_fraction': 0.85,
    'bagging_freq': 3,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'n_estimators': 3000,  # More trees for lower LR
    'verbose': -1,
    'random_state': 42
}

xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'learning_rate': 0.01,  # Lower for better accuracy
    'max_depth': 7,
    'min_child_weight': 20,
    'subsample': 0.85,
    'colsample_bytree': 0.75,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'n_estimators': 3000,  # More trees
    'tree_method': 'hist',
    'random_state': 42,
    'verbosity': 0
}

cat_params = {
    'iterations': 3000,
    'learning_rate': 0.01,  # Lower for better accuracy
    'depth': 7,
    'l2_leaf_reg': 3,
    'random_seed': 42,
    'verbose': 0,
    'eval_metric': 'AUC',
    'early_stopping_rounds': 200
}

# Random Forest parameters - increased
rf_params = {
    'n_estimators': 1000,  # More trees
    'max_depth': 18,
    'min_samples_split': 15,
    'min_samples_leaf': 8,
    'max_features': 'sqrt',
    'n_jobs': -1,
    'random_state': 42
}

# Extra Trees parameters - increased
et_params = {
    'n_estimators': 1000,  # More trees
    'max_depth': 18,
    'min_samples_split': 15,
    'min_samples_leaf': 8,
    'max_features': 'sqrt',
    'n_jobs': -1,
    'random_state': 42
}

# HistGradientBoosting parameters
from sklearn.ensemble import HistGradientBoostingClassifier
hgb_params = {
    'max_iter': 1000,
    'learning_rate': 0.02,
    'max_depth': 8,
    'min_samples_leaf': 20,
    'l2_regularization': 0.1,
    'random_state': 42
}

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

In [5]:
def engineer_features(df):    
    """Create engineered features based on domain knowledge - Enhanced version"""    
    df = df.copy()        
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')        
    df['tenure_bin'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72],                               
                              labels=['0-12', '12-24', '24-48', '48+'])
    df['tenure_bin'] = df['tenure_bin'].astype(str)    
    df['tenure_months'] = df['tenure']    
    df['tenure_squared'] = df['tenure'] ** 2    
    df['tenure_sqrt'] = np.sqrt(df['tenure'])    
    df['tenure_log'] = np.log1p(df['tenure'])    
    df['customer_value'] = df['tenure'] * df['MonthlyCharges']    
    df['customer_value_log'] = np.log1p(df['customer_value'])    
    df['avg_monthly_charges'] = np.where(df['tenure'] > 0,                                           
                                         df['TotalCharges'] / df['tenure'],                                           
                                         df['MonthlyCharges'])    
    df['charge_diff'] = df['MonthlyCharges'] - df['avg_monthly_charges']
    df['charge_ratio'] = df['MonthlyCharges'] / (df['avg_monthly_charges'] + 1)        
    service_cols = ['PhoneService', 'MultipleLines', 'InternetService',                     
                    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',                     
                    'TechSupport', 'StreamingTV', 'StreamingMovies']        
    df['service_count'] = sum((df[col] == 'Yes').astype(int) for col in service_cols)        
    df['has_fiber'] = (df['InternetService'] == 'Fiber optic').astype(int)    
    df['has_dsl'] = (df['InternetService'] == 'DSL').astype(int)    
    df['no_internet'] = (df['InternetService'] == 'No').astype(int)        
    df['has_security'] = ((df['OnlineSecurity'] == 'Yes') |                           
                          (df['TechSupport'] == 'Yes')).astype(int)    
    df['has_backup'] = ((df['OnlineBackup'] == 'Yes') |                         
                        (df['DeviceProtection'] == 'Yes')).astype(int)    
    df['has_streaming'] = ((df['StreamingTV'] == 'Yes') |                            
                           (df['StreamingMovies'] == 'Yes')).astype(int)    
    df['has_phone'] = (df['PhoneService'] == 'Yes').astype(int)    
    df['has_multiple_lines'] = (df['MultipleLines'] == 'Yes').astype(int)    
    df['no_protection'] = ((df['OnlineSecurity'] == 'No') &                            
                           (df['OnlineBackup'] == 'No') &                            
                           (df['DeviceProtection'] == 'No') &                           
                           (df['TechSupport'] == 'No')).astype(int)    
    df['full_protection'] = ((df['OnlineSecurity'] == 'Yes') |                              
                             (df['TechSupport'] == 'Yes')).astype(int)        
    df['is_month_to_month'] = (df['Contract'] == 'Month-to-month').astype(int)    
    df['is_electronic_check'] = (df['PaymentMethod'] == 'Electronic check').astype(int)    
    df['is_long_contract'] = ((df['Contract'] == 'One year') |                                
                              (df['Contract'] == 'Two year')).astype(int)        
    df['has_partner'] = (df['Partner'] == 'Yes').astype(int)    
    df['has_dependents'] = (df['Dependents'] == 'Yes').astype(int)    
    df['family_size'] = df['has_partner'] + df['has_dependents']    
    df['is_senior'] = df['SeniorCitizen']    
    df['is_paperless'] = (df['PaperlessBilling'] == 'Yes').astype(int)    
    df['high_monthly_charges'] = (df['MonthlyCharges'] > df['MonthlyCharges'].median()).astype(int)    
    df['new_customer_high_charge'] = ((df['tenure'] <= 12) &                                         
                                      (df['MonthlyCharges'] > df['MonthlyCharges'].median())).astype(int)    
    df['charges_per_service'] = df['MonthlyCharges'] / (df['service_count'] + 1)        
    df['new_customer'] = (df['tenure'] <= 6).astype(int)    
    df['mid_tenure'] = ((df['tenure'] > 6) & (df['tenure'] <= 24)).astype(int)    
    df['loyal_customer'] = (df['tenure'] > 48).astype(int)        
    df['month_to_month_fiber'] = df['is_month_to_month'] * df['has_fiber']    
    df['month_to_month_high_charge'] = df['is_month_to_month'] * df['high_monthly_charges']    
    df['new_customer_no_protection'] = df['new_customer'] * df['no_protection']    
    df['electronic_check_high_charge'] = df['is_electronic_check'] * df['high_monthly_charges']    
    df['senior_no_internet'] = df['is_senior'] * df['no_internet']        
    df['tenure_x_services'] = df['tenure'] * df['service_count']    
    df['tenure_x_security'] = df['tenure'] * df['has_security']    
    df['tenure_x_fiber'] = df['tenure'] * df['has_fiber']        
    df['customer_value_per_tenure'] = df['customer_value'] / (df['tenure'] + 1)    
    df['charges_vs_value'] = df['MonthlyCharges'] / (df['customer_value_per_tenure'] + 1)        
    return df


In [6]:
# Apply feature engineering
train_fe = engineer_features(train)
test_fe = engineer_features(test)

print(f"Train feature engineered shape: {train_fe.shape}")
print(f"Test feature engineered shape: {test_fe.shape}")

Train feature engineered shape: (601237, 66)
Test feature engineered shape: (254655, 65)


## 3. Preprocessing

In [7]:
# Handle TotalCharges
train_fe['TotalCharges'] = pd.to_numeric(train_fe['TotalCharges'], errors='coerce')
test_fe['TotalCharges'] = pd.to_numeric(test_fe['TotalCharges'], errors='coerce')

median_total = train_fe['TotalCharges'].median()
train_fe['TotalCharges'].fillna(median_total, inplace=True)
test_fe['TotalCharges'].fillna(median_total, inplace=True)

# Encode target
train_fe['Churn'] = (train_fe['Churn'] == 'Yes').astype(int)

# Feature columns
exclude_cols = ['id', 'Churn']
feature_cols = [c for c in train_fe.columns if c not in exclude_cols]

cat_cols_to_encode = train_fe[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = train_fe[feature_cols].select_dtypes(include=['int64', 'float64', 'int32']).columns.tolist()

print(f"Categorical columns: {len(cat_cols_to_encode)}")
print(f"Numerical columns: {len(num_cols)}")

Categorical columns: 16
Numerical columns: 48


In [8]:
# Label encode categorical columns
combined = pd.concat([train_fe[feature_cols], test_fe[feature_cols]], axis=0)

label_encoders = {}
for col in cat_cols_to_encode:
    le = LabelEncoder()
    combined[col] = le.fit_transform(combined[col].astype(str))
    label_encoders[col] = le

train_encoded = combined.iloc[:len(train_fe)]
test_encoded = combined.iloc[len(train_fe):]

# Prepare data
X = train_encoded.values
y = train_fe['Churn'].values
X_test = test_encoded.values
test_ids = test_fe['id'].values

# 9:1 split for validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

X_train: (541113, 64), X_val: (60124, 64), X_test: (254655, 64)


In [9]:
# Target Encoding for categorical features (using CV to avoid leakage)

print("Applying target encoding...")

# Columns to target encode
te_cols = ['Contract', 'PaymentMethod', 'InternetService', 'tenure_bin']

# Create a copy for target encoding
train_te = train_fe.copy()
test_te = test_fe.copy()

# Initialize target encoded columns
for col in te_cols:
    train_te[f'{col}_te'] = 0.0
    test_te[f'{col}_te'] = 0.0

# Use KFold for target encoding to avoid leakage
n_splits = 5
kfold = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for col in te_cols:
    train_te[f'{col}_te'] = 0.0
    global_mean = train_fe['Churn'].mean()
    
    # For training data, use OOF encoding
    for train_idx, val_idx in kfold.split(train_te, train_te['Churn']):
        train_fold = train_te.iloc[train_idx]
        val_fold = train_te.iloc[val_idx]
        
        # Calculate target mean for each category in training fold
        target_means = train_fold.groupby(col)['Churn'].mean()
        
        # Apply to validation fold
        train_te.loc[train_te.index[val_idx], f'{col}_te'] = train_te.iloc[val_idx][col].map(target_means).fillna(global_mean)
    
    # For test data, use full training data
    target_means_full = train_te.groupby(col)['Churn'].mean()
    test_te[f'{col}_te'] = test_te[col].map(target_means_full).fillna(global_mean)

# Add target encoded features to feature columns
te_feature_cols = [f'{col}_te' for col in te_cols]

# Update feature columns
feature_cols = feature_cols + te_feature_cols

# Update encoded data
train_encoded_with_te = train_encoded.copy()
test_encoded_with_te = test_encoded.copy()

for col in te_feature_cols:
    train_encoded_with_te[col] = train_te[col].values
    test_encoded_with_te[col] = test_te[col].values

# Update X, X_test with target encoded features
X = train_encoded_with_te.values
X_test = test_encoded_with_te.values

print(f"Added {len(te_feature_cols)} target encoded features")
print(f"X shape after target encoding: {X.shape}")

Applying target encoding...
Added 4 target encoded features
X shape after target encoding: (601237, 68)


## 4. Model Training - Stacking Ensemble

In [ ]:
# Train base models with CV and get OOF predictions
oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))
oof_rf = np.zeros(len(X))
oof_et = np.zeros(len(X))
oof_hgb = np.zeros(len(X))

test_lgb = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))
test_rf = np.zeros(len(X_test))
test_et = np.zeros(len(X_test))
test_hgb = np.zeros(len(X_test))

print("Training base models with 5-fold CV...")

# LightGBM with increased early stopping
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_vl = X[train_idx], X[val_idx]
    y_tr, y_vl = y[train_idx], y[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)],
              callbacks=[lgb.early_stopping(200, verbose=False)])
    
    oof_lgb[val_idx] = model.predict_proba(X_vl)[:, 1]
    test_lgb += model.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f"  Fold {fold+1} LightGBM done")

lgb_oof_auc = roc_auc_score(y, oof_lgb)
print(f"LightGBM OOF AUC: {lgb_oof_auc:.5f}")

# XGBoost with increased early stopping
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_vl = X[train_idx], X[val_idx]
    y_tr, y_vl = y[train_idx], y[val_idx]
    
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
    
    oof_xgb[val_idx] = model.predict_proba(X_vl)[:, 1]
    test_xgb += model.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f"  Fold {fold+1} XGBoost done")

xgb_oof_auc = roc_auc_score(y, oof_xgb)
print(f"XGBoost OOF AUC: {xgb_oof_auc:.5f}")

# CatBoost
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_vl = X[train_idx], X[val_idx]
    y_tr, y_vl = y[train_idx], y[val_idx]
    
    model = CatBoostClassifier(**cat_params)
    model.fit(X_tr, y_tr, eval_set=(X_vl, y_vl), use_best_model=True)
    
    oof_cat[val_idx] = model.predict_proba(X_vl)[:, 1]
    test_cat += model.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f"  Fold {fold+1} CatBoost done")

cat_oof_auc = roc_auc_score(y, oof_cat)
print(f"CatBoost OOF AUC: {cat_oof_auc:.5f}")

# RandomForest
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_vl = X[train_idx], X[val_idx]
    y_tr, y_vl = y[train_idx], y[val_idx]
    
    model = RandomForestClassifier(**rf_params)
    model.fit(X_tr, y_tr)
    
    oof_rf[val_idx] = model.predict_proba(X_vl)[:, 1]
    test_rf += model.predict_proba(X_test)[:, 1] / N_FOLDS

rf_oof_auc = roc_auc_score(y, oof_rf)
print(f"RandomForest OOF AUC: {rf_oof_auc:.5f}")

# ExtraTrees
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_vl = X[train_idx], X[val_idx]
    y_tr, y_vl = y[train_idx], y[val_idx]
    
    model = ExtraTreesClassifier(**et_params)
    model.fit(X_tr, y_tr)
    
    oof_et[val_idx] = model.predict_proba(X_vl)[:, 1]
    test_et += model.predict_proba(X_test)[:, 1] / N_FOLDS

et_oof_auc = roc_auc_score(y, oof_et)
print(f"ExtraTrees OOF AUC: {et_oof_auc:.5f}")

# HistGradientBoosting
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_vl = X[train_idx], X[val_idx]
    y_tr, y_vl = y[train_idx], y[val_idx]
    
    model = HistGradientBoostingClassifier(**hgb_params)
    model.fit(X_tr, y_tr)
    
    oof_hgb[val_idx] = model.predict_proba(X_vl)[:, 1]
    test_hgb += model.predict_proba(X_test)[:, 1] / N_FOLDS

hgb_oof_auc = roc_auc_score(y, oof_hgb)
print(f"HistGradientBoosting OOF AUC: {hgb_oof_auc:.5f}")

Training base models with 5-fold CV...
  Fold 1 LightGBM done
  Fold 2 LightGBM done
  Fold 3 LightGBM done
  Fold 4 LightGBM done
  Fold 5 LightGBM done
LightGBM OOF AUC: 0.91555
  Fold 1 XGBoost done


In [ ]:
# Meta-learner (stacking) with all 6 base models
oof_stack = np.column_stack([oof_lgb, oof_xgb, oof_cat, oof_rf, oof_et, oof_hgb])
test_stack = np.column_stack([test_lgb, test_xgb, test_cat, test_rf, test_et, test_hgb])

oof_meta = np.zeros(len(oof_stack))
test_meta = np.zeros(len(test_stack))

print("\nTraining meta-learner (Logistic Regression)...")
for fold, (train_idx, val_idx) in enumerate(skf.split(oof_stack, y)):
    X_tr, X_vl = oof_stack[train_idx], oof_stack[val_idx]
    y_tr, y_vl = y[train_idx], y[val_idx]
    
    meta_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    meta_model.fit(X_tr, y_tr)
    
    oof_meta[val_idx] = meta_model.predict_proba(X_vl)[:, 1]
    test_meta += meta_model.predict_proba(test_stack)[:, 1] / N_FOLDS

meta_auc = roc_auc_score(y, oof_meta)
print(f"Stacking Meta-Learner (LR) OOF AUC: {meta_auc:.5f}")

# Also try LightGBM as meta-learner
print("\nTraining meta-learner (LightGBM)...")
oof_meta_lgb = np.zeros(len(oof_stack))
test_meta_lgb = np.zeros(len(test_stack))

lgb_meta_params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 15,
    'max_depth': 4,
    'n_estimators': 300,
    'verbose': -1,
    'random_state': 42
}

for fold, (train_idx, val_idx) in enumerate(skf.split(oof_stack, y)):
    X_tr, X_vl = oof_stack[train_idx], oof_stack[val_idx]
    y_tr, y_vl = y[train_idx], y[val_idx]
    
    meta_lgb = lgb.LGBMClassifier(**lgb_meta_params)
    meta_lgb.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)],
                 callbacks=[lgb.early_stopping(50, verbose=False)])
    
    oof_meta_lgb[val_idx] = meta_lgb.predict_proba(X_vl)[:, 1]
    test_meta_lgb += meta_lgb.predict_proba(test_stack)[:, 1] / N_FOLDS

meta_lgb_auc = roc_auc_score(y, oof_meta_lgb)
print(f"Stacking Meta-Learner (LGB) OOF AUC: {meta_lgb_auc:.5f}")

# Try weighted average of both meta-learners
oof_meta_blend = 0.6 * oof_meta + 0.4 * oof_meta_lgb
test_meta_blend = 0.6 * test_meta + 0.4 * test_meta_lgb
meta_blend_auc = roc_auc_score(y, oof_meta_blend)
print(f"Blended Meta-Learner OOF AUC: {meta_blend_auc:.5f}")

# Simple average of all 6 base models
oof_avg6 = (oof_lgb + oof_xgb + oof_cat + oof_rf + oof_et + oof_hgb) / 6
test_avg6 = (test_lgb + test_xgb + test_cat + test_rf + test_et + test_hgb) / 6
avg6_auc = roc_auc_score(y, oof_avg6)
print(f"Simple Average (6 models) OOF AUC: {avg6_auc:.5f}")

# Choose best ensemble approach
results = {
    'LR Stacking': (meta_auc, test_meta),
    'LGB Stacking': (meta_lgb_auc, test_meta_lgb),
    'Blended Stacking': (meta_blend_auc, test_meta_blend),
    'Simple Average': (avg6_auc, test_avg6)
}

best_method = max(results.keys(), key=lambda k: results[k][0])
best_auc, best_test_pred = results[best_method]

print(f"\nBest ensemble method: {best_method} with AUC: {best_auc:.5f}")

# Use the best predictions
test_meta = best_test_pred

In [ ]:
# Save models for streaming prediction

print("\nSaving models for streaming prediction...")

# Train final models on full data for prediction
final_lgb = lgb.LGBMClassifier(**lgb_params)
final_lgb.fit(X, y)
with open(f'{MODEL_DIR}/lgb_model.pkl', 'wb') as f:
    pickle.dump(final_lgb, f)
print("LightGBM model saved.")

final_xgb = xgb.XGBClassifier(**xgb_params)
final_xgb.fit(X, y)
with open(f'{MODEL_DIR}/xgb_model.pkl', 'wb') as f:
    pickle.dump(final_xgb, f)
print("XGBoost model saved.")

final_cat = CatBoostClassifier(**cat_params)
final_cat.fit(X, y)
with open(f'{MODEL_DIR}/cat_model.pkl', 'wb') as f:
    pickle.dump(final_cat, f)
print("CatBoost model saved.")

# Save Random Forest
final_rf = RandomForestClassifier(**rf_params)
final_rf.fit(X, y)
with open(f'{MODEL_DIR}/rf_model.pkl', 'wb') as f:
    pickle.dump(final_rf, f)
print("RandomForest model saved.")

# Save Extra Trees
final_et = ExtraTreesClassifier(**et_params)
final_et.fit(X, y)
with open(f'{MODEL_DIR}/et_model.pkl', 'wb') as f:
    pickle.dump(final_et, f)
print("ExtraTrees model saved.")

# Save label encoders and feature columns
with open(f'{MODEL_DIR}/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)
with open(f'{MODEL_DIR}/feature_cols.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)
print("Label encoders and feature columns saved.")
print(f"All models saved to {MODEL_DIR}/")

## 5. Validation Performance Summary

In [ ]:
# Evaluate on held-out validation set
val_lgb = lgb.LGBMClassifier(**lgb_params)
val_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(150, verbose=False)])
val_pred_lgb = val_lgb.predict_proba(X_val)[:, 1]

val_xgb = xgb.XGBClassifier(**xgb_params)
val_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
val_pred_xgb = val_xgb.predict_proba(X_val)[:, 1]

val_cat = CatBoostClassifier(**cat_params)
val_cat.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
val_pred_cat = val_cat.predict_proba(X_val)[:, 1]

val_rf = RandomForestClassifier(**rf_params)
val_rf.fit(X_train, y_train)
val_pred_rf = val_rf.predict_proba(X_val)[:, 1]

val_et = ExtraTreesClassifier(**et_params)
val_et.fit(X_train, y_train)
val_pred_et = val_et.predict_proba(X_val)[:, 1]

val_hgb = HistGradientBoostingClassifier(**hgb_params)
val_hgb.fit(X_train, y_train)
val_pred_hgb = val_hgb.predict_proba(X_val)[:, 1]

# Use all 6 models in validation stack
val_stack = np.column_stack([val_pred_lgb, val_pred_xgb, val_pred_cat, val_pred_rf, val_pred_et, val_pred_hgb])
meta_model_final = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
meta_model_final.fit(oof_stack, y)
val_pred_meta = meta_model_final.predict_proba(val_stack)[:, 1]

val_auc = roc_auc_score(y_val, val_pred_meta)
print(f"Validation AUC-ROC: {val_auc:.5f}")

# Also check individual model validation performance
print("\nIndividual Model Validation AUC:")
print(f"  LightGBM:    {roc_auc_score(y_val, val_pred_lgb):.5f}")
print(f"  XGBoost:     {roc_auc_score(y_val, val_pred_xgb):.5f}")
print(f"  CatBoost:    {roc_auc_score(y_val, val_pred_cat):.5f}")
print(f"  RandomForest:{roc_auc_score(y_val, val_pred_rf):.5f}")
print(f"  ExtraTrees:  {roc_auc_score(y_val, val_pred_et):.5f}")

val_pred_binary = (val_pred_meta > 0.5).astype(int)
print("\nClassification Report (threshold=0.5):")
print(classification_report(y_val, val_pred_binary, target_names=['No Churn', 'Churn']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_val, val_pred_binary)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Churn', 'Churn'], 
            yticklabels=['No Churn', 'Churn'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix (AUC: {val_auc:.4f})')
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/confusion_matrix.png', dpi=100)
plt.show()

## 6. Prediction Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax1 = axes[0]
ax1.hist(val_pred_meta[y_val == 0], bins=50, alpha=0.7, label='No Churn', color='steelblue')
ax1.hist(val_pred_meta[y_val == 1], bins=50, alpha=0.7, label='Churn', color='coral')
ax1.set_xlabel('Predicted Probability')
ax1.set_ylabel('Count')
ax1.set_title('Validation Predictions by Actual Class')
ax1.legend()

ax2 = axes[1]
ax2.hist(test_meta, bins=50, alpha=0.7, color='green')
ax2.set_xlabel('Predicted Probability')
ax2.set_ylabel('Count')
ax2.set_title('Test Predictions Distribution')

plt.tight_layout()
plt.savefig(f'prediction_distributions.png', dpi=100)
plt.show()

print(f"\nTest predictions statistics:")
print(f"  Min: {test_meta.min():.4f}")
print(f"  Max: {test_meta.max():.4f}")
print(f"  Mean: {test_meta.mean():.4f}")
print(f"  Std: {test_meta.std():.4f}")

## 7. Streaming Row-by-Row Prediction

In [ ]:
# Load saved models for streaming prediction
print("Loading saved models...")
with open(f'{MODEL_DIR}/lgb_model.pkl', 'rb') as f:
    lgb_model = pickle.load(f)
with open(f'{MODEL_DIR}/xgb_model.pkl', 'rb') as f:
    xgb_model = pickle.load(f)
with open(f'{MODEL_DIR}/cat_model.pkl', 'rb') as f:
    cat_model = pickle.load(f)
with open(f'{MODEL_DIR}/rf_model.pkl', 'rb') as f:
    rf_model = pickle.load(f)
with open(f'{MODEL_DIR}/et_model.pkl', 'rb') as f:
    et_model = pickle.load(f)
with open(f'{MODEL_DIR}/label_encoders.pkl', 'rb') as f:
    loaded_label_encoders = pickle.load(f)
with open(f'{MODEL_DIR}/feature_cols.pkl', 'rb') as f:
    loaded_feature_cols = pickle.load(f)
print("All models loaded successfully.")

In [ ]:
# Streaming row-by-row prediction
def engineer_features_streaming(df):    
    """Create engineered features based on domain knowledge - streaming version"""    
    df = df.copy()        
    # Convert numeric columns that might be strings    
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')    
    df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'], errors='coerce')    
    df['tenure'] = pd.to_numeric(df['tenure'], errors='coerce').fillna(0).astype(int)        
    df['tenure_bin'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72], labels=['0-12', '12-24', '24-48', '48+']) 
    df['tenure_bin'] = df['tenure_bin'].astype(str)                         
    df['tenure_months'] = df['tenure']    
    df['customer_value'] = df['tenure'] * df['MonthlyCharges']    
    df['avg_monthly_charges'] = np.where(df['tenure'] > 0,                                           
                                         df['TotalCharges'] / df['tenure'],                                           
                                         df['MonthlyCharges'])        
    service_cols = ['PhoneService', 'MultipleLines', 'InternetService',                     
                    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',                     
                    'TechSupport', 'StreamingTV', 'StreamingMovies']        
    df['service_count'] = sum((df[col] == 'Yes').astype(int) for col in service_cols)        
    df['has_fiber'] = (df['InternetService'] == 'Fiber optic').astype(int)    
    df['has_dsl'] = (df['InternetService'] == 'DSL').astype(int)    
    df['no_internet'] = (df['InternetService'] == 'No').astype(int)        
    df['has_security'] = ((df['OnlineSecurity'] == 'Yes') |                           
                          (df['TechSupport'] == 'Yes')).astype(int)    
    df['has_backup'] = ((df['OnlineBackup'] == 'Yes') |                         
                        (df['DeviceProtection'] == 'Yes')).astype(int)    
    df['has_streaming'] = ((df['StreamingTV'] == 'Yes') |                            
                           (df['StreamingMovies'] == 'Yes')).astype(int)        
    df['is_month_to_month'] = (df['Contract'] == 'Month-to-month').astype(int)    
    df['is_electronic_check'] = (df['PaymentMethod'] == 'Electronic check').astype(int)    
    df['has_partner'] = (df['Partner'] == 'Yes').astype(int)    
    df['has_dependents'] = (df['Dependents'] == 'Yes').astype(int)    
    df['family_size'] = df['has_partner'] + df['has_dependents']    
    df['is_senior'] = df['SeniorCitizen']    
    df['is_paperless'] = (df['PaperlessBilling'] == 'Yes').astype(int)    
    df['high_monthly_charges'] = (df['MonthlyCharges'] > 70).astype(int)    
    df['new_customer_high_charge'] = ((df['tenure'] <= 12) &                                         
                                      (df['MonthlyCharges'] > 70)).astype(int)    
    df['charges_per_service'] = df['MonthlyCharges'] / (df['service_count'] + 1)        
    return df
    
def preprocess_row(row_df, label_encoders, feature_cols, median_total):    
    """Preprocess a single row for prediction"""    
    row_df['TotalCharges'] = pd.to_numeric(row_df['TotalCharges'], errors='coerce')    
    row_df['TotalCharges'].fillna(median_total, inplace=True)        
    for col in label_encoders:        
        if col in row_df.columns:            
            le = label_encoders[col]            
            val = str(row_df[col].iloc[0])            
            if val in le.classes_:                
                row_df[col] = le.transform([val])[0]            
            else:                
                row_df[col] = -1        
    X_row = row_df[feature_cols].values    
    return X_row
print("Starting streaming prediction...")m
edian_total = train_fe['TotalCharges'].median()
total_rows = 0
with open(TEST_INPUT, 'r') as infile, open(SUBMISSION_OUTPUT, 'w', newline='') as outfile:    
    reader = csv.DictReader(infile)    
    writer = csv.writer(outfile)        
    writer.writerow(['id', 'Churn'])        
    buffer_ids = []    
    buffer_rows = []    
    buffer_size = 100        
    for row in reader:        
        row_df = pd.DataFrame([row])        
        row_df = engineer_features_streaming(row_df)                
        try:            
            X_processed = preprocess_row(row_df, loaded_label_encoders, loaded_feature_cols, median_total)                        
            buffer_ids.append(int(row['id']))            
            buffer_rows.append(X_processed[0])                        
            if len(buffer_rows) >= buffer_size:                
                X_buffer = np.array(buffer_rows)                                
                pred_lgb = lgb_model.predict_proba(X_buffer)[:, 1]                
                pred_xgb = xgb_model.predict_proba(X_buffer)[:, 1]                
                pred_cat = cat_model.predict_proba(X_buffer)[:, 1]                
                pred_rf = rf_model.predict_proba(X_buffer)[:, 1]                
                pred_et = et_model.predict_proba(X_buffer)[:, 1]                                
                # Simple average of all 5 models                
                pred_meta = (pred_lgb + pred_xgb + pred_cat + pred_rf + pred_et) / 5                                
                for i, row_id in enumerate(buffer_ids):                    
                    writer.writerow([row_id, pred_meta[i]])                                
                    total_rows += len(buffer_ids)                
                print(f"Processed {total_rows} rows...")                                
                buffer_ids = []                
                buffer_rows = []                        
        except Exception as e:            
            print(f"Error processing row {row['id']}: {e}")            
            continue        
    if buffer_rows:        
        X_buffer = np.array(buffer_rows)                
        pred_lgb = lgb_model.predict_proba(X_buffer)[:, 1]        
        pred_xgb = xgb_model.predict_proba(X_buffer)[:, 1]        
        pred_cat = cat_model.predict_proba(X_buffer)[:, 1]        
        pred_rf = rf_model.predict_proba(X_buffer)[:, 1]        
        pred_et = et_model.predict_proba(X_buffer)[:, 1]                
        pred_meta = (pred_lgb + pred_xgb + pred_cat + pred_rf + pred_et) / 5                
        for i, row_id in enumerate(buffer_ids):            
            writer.writerow([row_id, pred_meta[i]])                
            total_rows += len(buffer_ids)        
        print(f"Processed remaining {len(buffer_ids)} rows.")
print(f"\nStreaming prediction complete! Total rows: {total_rows}")
print(f"Submission saved to {SUBMISSION_OUTPUT}")

In [ ]:
# Verify streaming submission matches batch submission
streaming_sub = pd.read_csv(SUBMISSION_OUTPUT)
print(f"Streaming submission shape: {streaming_sub.shape}")
print(f"Batch submission shape: {submission.shape}")

streaming_sub = streaming_sub.sort_values('id').reset_index(drop=True)
submission_sorted = submission.sort_values('id').reset_index(drop=True)

diff = np.abs(streaming_sub['Churn'].values - submission_sorted['Churn'].values)
print(f"Max difference in predictions: {diff.max():.6f}")
print(f"Mean difference in predictions: {diff.mean():.6f}")
print("\nStreaming submission sample:")
print(streaming_sub.head(10))

## 8. Summary

In [ ]:
print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"\nModel: Stacking Ensemble")
print(f"  - Base Models: LightGBM, XGBoost, CatBoost")
print(f"  - Meta-Learner: Logistic Regression")
print(f"\nBase Model Performance (5-Fold CV OOF AUC):")
print(f"  - LightGBM:  {lgb_oof_auc:.5f}")
print(f"  - XGBoost:   {xgb_oof_auc:.5f}")
print(f"  - CatBoost:  {cat_oof_auc:.5f}")
print(f"  - Stacking:  {meta_auc:.5f}")
print(f"\nValidation Performance (10% holdout):")
print(f"  - AUC-ROC: {val_auc:.5f}")
print(f"\nSubmission:")
print(f"  - File: {DATA_DIR}/submission.csv")
print(f"  - Rows: {total_rows}")
print(f"  - Prediction range: [{test_meta.min():.4f}, {test_meta.max():.4f}]")
print("=" * 60)

In [ ]:
# Optimized Blending - Find best weights using grid search


print("\nOptimizing blend weights...")

def blend_score(weights, preds, y_true):
    """Calculate negative AUC for optimization (minimize = maximize AUC)"""
    weights = np.array(weights)
    weights = weights / weights.sum()  # Normalize
    blended = sum(w * p for w, p in zip(weights, preds))
    return -roc_auc_score(y_true, blended)

# All 6 base model predictions
oof_preds = [oof_lgb, oof_xgb, oof_cat, oof_rf, oof_et, oof_hgb]
test_preds = [test_lgb, test_xgb, test_cat, test_rf, test_et, test_hgb]

# Grid search for optimal weights
best_score = 0
best_weights = None

# Try different weight combinations
for w1 in np.arange(0.1, 0.4, 0.05):
    for w2 in np.arange(0.1, 0.4, 0.05):
        for w3 in np.arange(0.1, 0.4, 0.05):
            for w4 in np.arange(0.0, 0.2, 0.05):
                for w5 in np.arange(0.0, 0.2, 0.05):
                    w6 = 1.0 - w1 - w2 - w3 - w4 - w5
                    if w6 < 0 or w6 > 0.3:
                        continue
                    weights = [w1, w2, w3, w4, w5, w6]
                    blended = sum(w * p for w, p in zip(weights, oof_preds))
                    score = roc_auc_score(y, blended)
                    if score > best_score:
                        best_score = score
                        best_weights = weights

print(f"Optimized weights: LGB={best_weights[0]:.2f}, XGB={best_weights[1]:.2f}, CAT={best_weights[2]:.2f}, RF={best_weights[3]:.2f}, ET={best_weights[4]:.2f}, HGB={best_weights[5]:.2f}")
print(f"Optimized blend OOF AUC: {best_score:.5f}")

# Create optimized blend predictions
oof_optimized = sum(w * p for w, p in zip(best_weights, oof_preds))
test_optimized = sum(w * p for w, p in zip(best_weights, test_preds))

# Compare with previous best
if best_score > best_auc:
    print("Using optimized blend for final predictions!")
    test_meta = test_optimized
    best_auc = best_score
else:
    print("Keeping previous best ensemble (stacking performed better)")

print(f"Final OOF AUC: {best_auc:.5f}")

## Potential Improvements

1. **Feature Engineering**: Add more interaction features, polynomial features
2. **Hyperparameter Tuning**: Use Optuna or GridSearch for better hyperparameters
3. **More Base Models**: Add RandomForest, ExtraTrees, Neural Networks
4. **Different Meta-Learner**: Try LightGBM as meta-learner instead of Logistic Regression
5. **Weighted Stacking**: Optimize weights for base model predictions
6. **Use Original Data**: Incorporate original IBM dataset for training
7. **Threshold Optimization**: Find optimal classification threshold for business metrics